# ML-08 — Capstone Modeling Lane

**Lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026  
**Primary question:** Can a simple learned model improve ranking quality over the transparent Week-4 action-score baseline?

### Method choice

This notebook starts with **Logistic Regression** because the target is binary (future decline vs. no future decline), the features are tabular March-only signals, and the model produces interpretable probabilities. The model is intentionally simple: complexity is not rewarded unless it improves decision support.

The ML-08 population follows the **established Week-4 data contract**: only March and April rows with `gsc_data_available IS TRUE` are eligible. The Week-4 baseline is reproduced as an already-established rule: its CTR cutoff is calculated once from the full valid March evaluation population, before the ML-08 train/test split, and is then frozen for the comparison. This preserves the baseline rather than silently retraining or changing it for ML-08.

**Primary metric:** Precision@K for K = 10, 50, 100, 500, because this lane is a prioritization/ranking problem. ROC-AUC and average precision are secondary diagnostics.


In [1]:
%pip -q install duckdb pandas numpy scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

MARCH_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"

RANDOM_STATE = 42
print("Warehouse connection configured.")
print("Decision window: March 2026 | Outcome window: April 2026")

Warehouse connection configured.
Decision window: March 2026 | Outcome window: April 2026


## 1. Method choice and why

**Chosen method: Logistic Regression.**

It fits the binary outcome and gives a continuous probability that can rank pages for review. It is a strong first learned model for this lane because it is interpretable, reproducible, and much less complex than an ensemble. If it cannot materially beat the transparent rule, adding complexity would not be justified for this assignment.

The model uses only information available at the March decision point. April is used only to construct the evaluation label.

In [2]:
march_sql = f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,
    CASE WHEN SUM(gsc_impressions) > 0
         THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
         ELSE NULL END AS march_ctr_pct,
    CASE WHEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                       THEN gsc_impressions ELSE 0 END) > 0
         THEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                       THEN gsc_impressions * gsc_avg_position ELSE 0 END)
              / SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                         THEN gsc_impressions ELSE 0 END)
         ELSE NULL END AS march_avg_position,
    COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_impression_days
FROM read_parquet('{MARCH_REL}')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''

april_sql = f'''
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions
FROM read_parquet('{APRIL_REL}')
WHERE month = '2026-04'
  AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''

march = con.execute(march_sql).df()
april = con.execute(april_sql).df()

key_cols = ["client_hash_id", "content_hash_id"]
assert not march.duplicated(key_cols).any(), "March aggregation produced duplicate keys."
assert not april.duplicated(key_cols).any(), "April aggregation produced duplicate keys."

df = march.merge(april, on=key_cols, how="inner", validate="one_to_one")

df["future_decline_label"] = (
    (df["march_impressions"] > 0)
    & (df["april_impressions"] < 0.80 * df["march_impressions"])
).astype(int)

# Freeze the established Week-4 CTR cutoff on the full valid March evaluation population.
# This is deliberately done BEFORE the ML-08 split and is not fitted on train/test rows.
baseline_band = df.loc[
    (df["march_impressions"] >= 500)
    & df["march_avg_position"].between(4, 20, inclusive="both")
    & df["march_ctr_pct"].notna()
].copy()

ctr_cutoff = baseline_band["march_ctr_pct"].median()
assert pd.notna(ctr_cutoff), "Could not reproduce the established Week-4 CTR cutoff."

print(f"Valid Week-4/ML-08 evaluation population: {len(df):,}")
print(f"Observed future-decline rate: {df['future_decline_label'].mean():.4f}")
print(f"Frozen Week-4 CTR cutoff: {ctr_cutoff:.4f}%")
display(df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Valid Week-4/ML-08 evaluation population: 158,549
Observed future-decline rate: 0.4782
Frozen Week-4 CTR cutoff: 0.1908%


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,april_impressions,future_decline_label
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.742857,24,27.0,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.049866,31,16121.0,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,6.410714,27,81.0,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,5.872159,31,213.0,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,15.276596,21,23.0,1


## 2. Split design

ML-08 uses one fixed, stratified 80/20 train/test split. The learned model is fitted on train only and evaluated on test only.

The Week-4 baseline is **not retrained on the split**. Its already-established CTR cutoff was frozen above from the full valid March evaluation population before this split, so the baseline used below is the historical Week-4 rule rather than a newly optimized rule.

Client/content identifiers stay out of the predictive feature matrix. The split is row-based rather than grouped by client; this is acceptable for this assignment's single held-out comparison, but the notebook reports client overlap as a validation limitation rather than hiding it.


In [3]:
FEATURES = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_impression_days",
]
TARGET = "future_decline_label"

assert not any(c in FEATURES for c in [
    "april_impressions", "future_decline_label", "trend_direction", "trend_pct"
]), "Leakage feature detected."

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df[TARGET],
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")
print(f"Train decline rate: {train[TARGET].mean():.4f}")
print(f"Test decline rate:  {test[TARGET].mean():.4f}")
print(f"Client overlap in this ML-08 split: {len(set(train.client_hash_id) & set(test.client_hash_id)):,}")

X_train = train[FEATURES].copy()
X_test = test[FEATURES].copy()
y_train = train[TARGET].copy()
y_test = test[TARGET].copy()

Train rows: 126,839
Test rows:  31,710
Train decline rate: 0.4782
Test decline rate:  0.4781
Client overlap in this ML-08 split: 45


## 3. Train + compare vs my baseline

The learned model is trained only on the training partition. The Week-4 action score uses the **frozen Week-4 CTR cutoff** established before the split. Both methods are ranked on the exact same held-out test rows.

The baseline rule remains transparent:
- high-volume signal: March impressions ≥ 500
- CTR-fix signal: position 4–20 and CTR below the frozen Week-4 cutoff in that visible band
- score 5 for both, 3 for high-volume only, 0 otherwise


In [4]:
# Reproduce the established Week-4 baseline on the held-out test set.
# Important: ctr_cutoff was frozen from the full valid evaluation population BEFORE the split.
test["baseline_high_volume"] = test["march_impressions"] >= 500
test["baseline_ctr_fix"] = (
    test["baseline_high_volume"]
    & test["march_avg_position"].between(4, 20, inclusive="both")
    & test["march_ctr_pct"].notna()
    & (test["march_ctr_pct"] < ctr_cutoff)
)
test["baseline_score"] = np.select(
    [
        test["baseline_high_volume"] & test["baseline_ctr_fix"],
        test["baseline_high_volume"],
    ],
    [5, 3],
    default=0,
).astype(float)

# Train Logistic Regression.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
test["model_probability"] = model.predict_proba(X_test)[:, 1]

def precision_at_k(frame, score_col, k):
    ranked = frame.sort_values(
        [score_col, "march_impressions", "march_clicks"],
        ascending=[False, False, False],
    )
    top = ranked.head(min(k, len(ranked)))
    return float(top[TARGET].mean()) if len(top) else np.nan

ks = [10, 50, 100, 500]
rows = []
for method, score_col in [
    ("Week-4 baseline", "baseline_score"),
    ("Logistic Regression", "model_probability"),
]:
    for k in ks:
        rows.append({
            "method": method,
            "k": k,
            "precision_at_k": precision_at_k(test, score_col, k),
        })

comparison = pd.DataFrame(rows)
comparison["precision_pct"] = (100 * comparison["precision_at_k"]).round(2)
display(comparison)

secondary = pd.DataFrame([
    {
        "method": "Week-4 baseline",
        "ROC_AUC": roc_auc_score(y_test, test["baseline_score"]),
        "Average_Precision": average_precision_score(y_test, test["baseline_score"]),
    },
    {
        "method": "Logistic Regression",
        "ROC_AUC": roc_auc_score(y_test, test["model_probability"]),
        "Average_Precision": average_precision_score(y_test, test["model_probability"]),
    },
])
display(secondary.round(4))


,method,k,precision_at_k,precision_pct
0,Week-4 baseline,10,0.500,50.0
1,Week-4 baseline,50,0.480,48.0
2,Week-4 baseline,100,0.500,50.0
3,Week-4 baseline,500,0.530,53.0
4,Logistic Regression,10,0.600,60.0
5,Logistic Regression,50,0.640,64.0
6,Logistic Regression,100,0.650,65.0
7,Logistic Regression,500,0.672,67.2


,method,ROC_AUC,Average_Precision
0,Week-4 baseline,0.5221,0.4987
1,Logistic Regression,0.6400,0.6003


## 4. Errors and interpretation

A useful model is not just a score. We inspect false positives, false negatives, and the features the model relies on. Feature interpretation uses permutation importance on the held-out test set; this is descriptive, not causal.

In [5]:
test["predicted_class"] = (test["model_probability"] >= 0.50).astype(int)

false_positives = test[
    (test["predicted_class"] == 1) & (test[TARGET] == 0)
].copy()

false_negatives = test[
    (test["predicted_class"] == 0) & (test[TARGET] == 1)
].copy()

error_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_impression_days",
    "model_probability",
    TARGET,
]

print(f"False positives: {len(false_positives):,}")
print(f"False negatives: {len(false_negatives):,}")

print("\nThree concrete false-positive examples:")
display(
    false_positives.sort_values("model_probability", ascending=False)[error_cols].head(3)
)

print("Three concrete false-negative examples:")
display(
    false_negatives.sort_values("model_probability", ascending=True)[error_cols].head(3)
)

perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
)

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance)

baseline_100 = comparison.loc[
    (comparison["method"] == "Week-4 baseline") & (comparison["k"] == 100),
    "precision_at_k",
].iloc[0]

model_100 = comparison.loc[
    (comparison["method"] == "Logistic Regression") & (comparison["k"] == 100),
    "precision_at_k",
].iloc[0]

print("\nInterpretation")
print(
    f"At K=100, Logistic Regression precision is {100*model_100:.2f}% "
    f"versus {100*baseline_100:.2f}% for the Week-4 baseline."
)

if model_100 > baseline_100:
    print("The learned model provides measured improvement at K=100 on this held-out split.")
elif model_100 < baseline_100:
    print("The transparent baseline remains stronger at K=100 on this held-out split; extra complexity is not justified by this result alone.")
else:
    print("The learned model and baseline are tied at K=100 on this held-out split.")

print("Permutation importance describes predictive reliance; it does not establish causality.")

False positives: 7,377
False negatives: 5,350

Three concrete false-positive examples:


,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,model_probability,future_decline_label
68355,63201.0,2.0,0.003165,8.707520,31,0.828193,0
133010,69600.0,10.0,0.014368,29.996351,31,0.808748,0
25724,83293.0,22.0,0.026413,29.833095,31,0.808447,0


Three concrete false-negative examples:


,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,model_probability,future_decline_label
60841,130338.0,1526.0,1.170802,3.176434,31,2.058449e-14,1
65456,245276.0,1480.0,0.603402,2.757730,29,5.065349e-13,1
87638,106025.0,663.0,0.625324,3.156491,31,3.737243e-06,1


,feature,importance_mean,importance_std
4,march_impression_days,0.089009,0.002256
1,march_clicks,0.077083,0.000748
3,march_avg_position,0.014160,0.001710
0,march_impressions,0.013057,0.001522
2,march_ctr_pct,0.000693,0.000240



Interpretation
At K=100, Logistic Regression precision is 65.00% versus 50.00% for the Week-4 baseline.
The learned model provides measured improvement at K=100 on this held-out split.
Permutation importance describes predictive reliance; it does not establish causality.


## Self-check

- [x] Method choice is explicit and justified.
- [x] The Week-4/ML-08 population is restricted to rows where `gsc_data_available IS TRUE` for both March and April.
- [x] The Week-4 baseline CTR cutoff is frozen from the full valid March evaluation population before the ML-08 split.
- [x] A fixed, stratified train/test split is used.
- [x] The learned model sees March-only features; April is label-only.
- [x] Label-derived/future fields are excluded from features.
- [x] Week-4 baseline and model are evaluated on the same held-out rows and the same Precision@K metrics.
- [x] ROC-AUC and average precision are secondary diagnostics, not the decision rule.
- [x] False positives and false negatives are inspected.
- [x] Permutation importance is reported with a non-causal interpretation.
- [x] Run the notebook top-to-bottom in Colab with your `HF_TOKEN`, inspect the final outputs, then commit the **executed** notebook to `work/notebooks/w05_model.ipynb`.


